In [15]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

pd.set_option("display.max_colwidth", 200)


In [19]:
!pip -q install datasets


In [21]:
from datasets import load_dataset

dataset = load_dataset("papluca/language-identification")


README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/12.0M [00:00<?, ?B/s]

valid.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/70000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [22]:
from datasets import load_dataset
import pandas as pd

# تحميل الداتا مباشرة بعد تفعيل الإنترنت
dataset = load_dataset("papluca/language-identification")

# تحويلها إلى Pandas DataFrames
train_df = pd.DataFrame(dataset['train'])
val_df = pd.DataFrame(dataset['validation'])
test_df = pd.DataFrame(dataset['test'])

print("تم تحميل الداتا بنجاح!")
print("حجم ملف التدريب (Train):", train_df.shape)
print(train_df.head(3))

تم تحميل الداتا بنجاح!
حجم ملف التدريب (Train): (70000, 2)
  labels  \
0     pt   
1     bg   
2     zh   

                                                                                                                                                                text  
0  os chefes de defesa da estónia, letónia, lituânia, alemanha, itália, espanha e eslováquia assinarão o acordo para fornecer pessoal e financiamento para o centro.  
1                                размерът на хоризонталната мрежа може да бъде по реда на няколко километра ( km ) за на симулация до около 100 km за на симулация .  
2                                                   很好，以前从不去评价，不知道浪费了多少积分，现在知道积分可以换钱，就要好好评价了，后来我就把这段话复制走了，既能赚积分，还省事，走到哪复制到哪，最重要的是，不用认真的评论了，不用想还差多少字，直接发出就可以了，推荐给大家！！  


In [23]:
print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

print("\nNumber of languages:")
print(train_df["labels"].nunique())

print("\nLanguages:")
print(sorted(train_df["labels"].unique()))

print("\nMissing values:")
print(train_df.isnull().sum())

print("\nDuplicate rows:")
print(train_df.duplicated().sum())


Train shape: (70000, 2)
Validation shape: (10000, 2)
Test shape: (10000, 2)

Columns:
['labels', 'text']

Number of languages:
20

Languages:
['ar', 'bg', 'de', 'el', 'en', 'es', 'fr', 'hi', 'it', 'ja', 'nl', 'pl', 'pt', 'ru', 'sw', 'th', 'tr', 'ur', 'vi', 'zh']

Missing values:
labels    0
text      0
dtype: int64

Duplicate rows:
1020


In [25]:
print(train_df["labels"].value_counts())


labels
pt    3500
bg    3500
zh    3500
th    3500
ru    3500
pl    3500
ur    3500
sw    3500
tr    3500
es    3500
ar    3500
it    3500
hi    3500
de    3500
el    3500
nl    3500
fr    3500
vi    3500
en    3500
ja    3500
Name: count, dtype: int64


In [26]:
# Inspect duplicate samples

duplicates = train_df[train_df.duplicated(subset=["text"], keep=False)] \
    .sort_values("text")

print("Number of duplicated text rows:", len(duplicates))

print("\nExamples:")
display(duplicates.head(20))


Number of duplicated text rows: 1851

Examples:


,labels,text
13778,ar,- ولدى المجلس حاليا مشروع نشط لمعالجة معايير الموارد الطبيعية .
28542,ar,- ولدى المجلس حاليا مشروع نشط لمعالجة معايير الموارد الطبيعية .
24676,hi,1822 में शहर की अंग ् रेजी कॉलोनी के वित ् त-पोषण के लिए निवासियों और आगंतुकों की एक तरह की सैर की गई है .
44424,hi,1822 में शहर की अंग ् रेजी कॉलोनी के वित ् त-पोषण के लिए निवासियों और आगंतुकों की एक तरह की सैर की गई है .
1686,tr,"2001 ' in ikinci yarısında 183,000 ' den fazla kişinin görev yaptığını belirttiler ."
32904,tr,"2001 ' in ikinci yarısında 183,000 ' den fazla kişinin görev yaptığını belirttiler ."
35261,nl,"49 doden, 148 gewonden bij gewelddadige aanvallen in Irak..."
8775,nl,"49 doden, 148 gewonden bij gewelddadige aanvallen in Irak..."
40619,it,"49 morti, 148 feriti in violenti attacchi in Iraq"
35372,it,"49 morti, 148 feriti in violenti attacchi in Iraq"


In [27]:
# Check whether duplicated texts have conflicting labels

duplicate_label_counts = (
    duplicates.groupby("text")["labels"]
    .nunique()
)

print("Duplicated texts with more than one label:",
      (duplicate_label_counts > 1).sum())


Duplicated texts with more than one label: 2


In [28]:
# Show texts that appear with different labels

conflicting_texts = (
    train_df.groupby("text")["labels"]
    .nunique()
)

conflicting_texts = conflicting_texts[conflicting_texts > 1]

print("Number of conflicting texts:", len(conflicting_texts))

for text in conflicting_texts.index:
    print("\nTEXT:")
    print(text)
    
    print("\nLABELS:")
    print(train_df[train_df["text"] == text][["labels", "text"]].to_string(index=False))


Number of conflicting texts: 2

TEXT:
Santorum Romping To Minnesota Victory

LABELS:
labels                                  text
    pt Santorum Romping To Minnesota Victory
    nl Santorum Romping To Minnesota Victory

TEXT:
Snowden Hits Hurdles in Search for Asylum

LABELS:
labels                                      text
    pl Snowden Hits Hurdles in Search for Asylum
    nl Snowden Hits Hurdles in Search for Asylum


In [29]:
# Remove exact duplicate texts from the training set
train_df = train_df.drop_duplicates(subset=["text"], keep="first").reset_index(drop=True)

# Remove texts that have conflicting language labels
label_counts = train_df.groupby("text")["labels"].nunique()
conflicting_texts = label_counts[label_counts > 1].index

train_df = train_df[
    ~train_df["text"].isin(conflicting_texts)
].reset_index(drop=True)

print("Training shape after cleaning:", train_df.shape)
print("Remaining duplicate texts:", train_df.duplicated(subset=["text"]).sum())
print("Remaining conflicting texts:",
      train_df.groupby("text")["labels"].nunique().gt(1).sum())



Training shape after cleaning: (68978, 2)
Remaining duplicate texts: 0
Remaining conflicting texts: 0


In [30]:
# Inspect raw examples before preprocessing

for i in range(5):
    print(f"Label: {train_df.iloc[i]['labels']}")
    print(f"Text : {train_df.iloc[i]['text']}")
    print("-" * 80)



Label: pt
Text : os chefes de defesa da estónia, letónia, lituânia, alemanha, itália, espanha e eslováquia assinarão o acordo para fornecer pessoal e financiamento para o centro.
--------------------------------------------------------------------------------
Label: bg
Text : размерът на хоризонталната мрежа може да бъде по реда на няколко километра ( km ) за на симулация до около 100 km за на симулация .
--------------------------------------------------------------------------------
Label: zh
Text : 很好，以前从不去评价，不知道浪费了多少积分，现在知道积分可以换钱，就要好好评价了，后来我就把这段话复制走了，既能赚积分，还省事，走到哪复制到哪，最重要的是，不用认真的评论了，不用想还差多少字，直接发出就可以了，推荐给大家！！
--------------------------------------------------------------------------------
Label: th
Text : สำหรับ ของเก่า ที่ จริงจัง ลอง   honeychurch   ของเก่า ที่ ไม่   29   สำหรับ เฟอร์นิเจอร์ และ เงิน ไท ร้อง บริษัท ที่   122   สำหรับ ลาย คราม
--------------------------------------------------------------------------------
Label: ru
Text : Он увеличил давление .
-------------------

In [31]:
import re

def preprocess_text(text):
    # 1. Lowercasing
    text = text.lower()

    # 2. Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # 3. Remove newlines and tabs
    text = text.replace('\n', ' ').replace('\t', ' ')

    # 4. Remove extra whitespaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# Apply preprocessing
train_df["clean_text"] = train_df["text"].apply(preprocess_text)
val_df["clean_text"] = val_df["text"].apply(preprocess_text)
test_df["clean_text"] = test_df["text"].apply(preprocess_text)

print("Preprocessing completed successfully.")


Preprocessing completed successfully.


In [32]:
for i in range(5):
    print("Label:", train_df.iloc[i]["labels"])
    print("Before:", train_df.iloc[i]["text"])
    print("After :", train_df.iloc[i]["clean_text"])
    print("-" * 100)


Label: pt
Before: os chefes de defesa da estónia, letónia, lituânia, alemanha, itália, espanha e eslováquia assinarão o acordo para fornecer pessoal e financiamento para o centro.
After : os chefes de defesa da estónia, letónia, lituânia, alemanha, itália, espanha e eslováquia assinarão o acordo para fornecer pessoal e financiamento para o centro.
----------------------------------------------------------------------------------------------------
Label: bg
Before: размерът на хоризонталната мрежа може да бъде по реда на няколко километра ( km ) за на симулация до около 100 km за на симулация .
After : размерът на хоризонталната мрежа може да бъде по реда на няколко километра ( km ) за на симулация до около 100 km за на симулация .
----------------------------------------------------------------------------------------------------
Label: zh
Before: 很好，以前从不去评价，不知道浪费了多少积分，现在知道积分可以换钱，就要好好评价了，后来我就把这段话复制走了，既能赚积分，还省事，走到哪复制到哪，最重要的是，不用认真的评论了，不用想还差多少字，直接发出就可以了，推荐给大家！！
After : 很好，以前从不去评价，不知道浪费了多少

In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF with word unigrams + bigrams
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

X_train = vectorizer.fit_transform(train_df["text"])
X_val = vectorizer.transform(val_df["text"])
X_test = vectorizer.transform(test_df["text"])

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))


X_train shape: (68978, 176649)
X_val shape: (10000, 176649)
X_test shape: (10000, 176649)
Vocabulary size: 176649


In [34]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Create the classifier
clf = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train
clf.fit(X_train, train_df["labels"])

print("Model training completed!")


Model training completed!


In [35]:
# Predict on validation set
y_val_pred = clf.predict(X_val)

# Evaluation
val_accuracy = accuracy_score(val_df["labels"], y_val_pred)

print("Validation Accuracy:", round(val_accuracy, 4))
print("\nClassification Report:")
print(classification_report(
    val_df["labels"],
    y_val_pred,
    digits=4
))


Validation Accuracy: 0.9159

Classification Report:
              precision    recall  f1-score   support

          ar     1.0000    0.8820    0.9373       500
          bg     0.9877    0.9600    0.9736       500
          de     1.0000    0.9740    0.9868       500
          el     1.0000    0.9900    0.9950       500
          en     0.9819    0.9760    0.9789       500
          es     0.9821    0.9860    0.9840       500
          fr     1.0000    0.9860    0.9930       500
          hi     1.0000    0.9400    0.9691       500
          it     0.9880    0.9880    0.9880       500
          ja     0.3952    0.9960    0.5659       500
          nl     0.9920    0.9860    0.9890       500
          pl     0.9815    0.9560    0.9686       500
          pt     0.9880    0.9860    0.9870       500
          ru     0.9956    0.9100    0.9509       500
          sw     0.9918    0.9620    0.9766       500
          th     1.0000    0.6840    0.8124       500
          tr     0.9753    0.

In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

# Character-level TF-IDF
char_vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.95
)

# Transform data
X_train_char = char_vectorizer.fit_transform(train_df["text"])
X_val_char = char_vectorizer.transform(val_df["text"])

print("X_train_char shape:", X_train_char.shape)
print("X_val_char shape:", X_val_char.shape)
print("Character vocabulary size:", len(char_vectorizer.vocabulary_))


X_train_char shape: (68978, 535640)
X_val_char shape: (10000, 535640)
Character vocabulary size: 535640


In [37]:
from sklearn.svm import LinearSVC

# Enhanced classifier
char_clf = LinearSVC()

# Train
char_clf.fit(X_train_char, train_df["labels"])

print("Enhanced model training completed!")


Enhanced model training completed!


In [38]:
from sklearn.metrics import accuracy_score, classification_report

# Predictions on validation set
y_val_pred_char = char_clf.predict(X_val_char)

# Accuracy
char_val_accuracy = accuracy_score(
    val_df["labels"],
    y_val_pred_char
)

print("Enhanced Validation Accuracy:", round(char_val_accuracy, 4))

print("\nEnhanced Classification Report:")
print(
    classification_report(
        val_df["labels"],
        y_val_pred_char,
        digits=4
    )
)


Enhanced Validation Accuracy: 0.9949

Enhanced Classification Report:
              precision    recall  f1-score   support

          ar     1.0000    0.9920    0.9960       500
          bg     1.0000    1.0000    1.0000       500
          de     1.0000    0.9960    0.9980       500
          el     1.0000    1.0000    1.0000       500
          en     0.9901    1.0000    0.9950       500
          es     0.9980    1.0000    0.9990       500
          fr     0.9980    1.0000    0.9990       500
          hi     1.0000    0.9500    0.9744       500
          it     0.9960    1.0000    0.9980       500
          ja     1.0000    0.9980    0.9990       500
          nl     0.9901    1.0000    0.9950       500
          pl     1.0000    0.9980    0.9990       500
          pt     1.0000    1.0000    1.0000       500
          ru     1.0000    1.0000    1.0000       500
          sw     0.9653    1.0000    0.9823       500
          th     1.0000    0.9960    0.9980       500
          t

In [39]:
# Transform test data using the fitted character TF-IDF vectorizer
X_test_char = char_vectorizer.transform(test_df["text"])

# Final predictions
y_test_pred = char_clf.predict(X_test_char)

# Final evaluation
test_accuracy = accuracy_score(
    test_df["labels"],
    y_test_pred
)

print("Final Test Accuracy:", round(test_accuracy, 4))

print("\nFinal Test Classification Report:")
print(
    classification_report(
        test_df["labels"],
        y_test_pred,
        digits=4
    )
)


Final Test Accuracy: 0.9947

Final Test Classification Report:
              precision    recall  f1-score   support

          ar     1.0000    0.9980    0.9990       500
          bg     0.9980    1.0000    0.9990       500
          de     1.0000    0.9960    0.9980       500
          el     1.0000    1.0000    1.0000       500
          en     0.9980    1.0000    0.9990       500
          es     0.9940    1.0000    0.9970       500
          fr     1.0000    1.0000    1.0000       500
          hi     1.0000    0.9640    0.9817       500
          it     0.9940    0.9920    0.9930       500
          ja     1.0000    0.9980    0.9990       500
          nl     0.9940    1.0000    0.9970       500
          pl     1.0000    0.9980    0.9990       500
          pt     0.9940    0.9940    0.9940       500
          ru     1.0000    0.9980    0.9990       500
          sw     0.9651    0.9960    0.9803       500
          th     1.0000    0.9960    0.9980       500
          tr     0